<img src="https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg" width="200">

# Module M2.10 — Graduated autonomy & human-in-the-loop agent design

This super-notebook is the hands-on companion to PDS Module 2.10. You will work through three connected pieces, in one continuous arc, in about 45 minutes:

1. **Setup** — paste your API key and confirm the SDKs are installed (~5 min)
2. **Basic tool-using agent** — the agent-loop foundation that the rest of this module layers governance on top of (~15 min). If you completed M2.9 already, you can skim this section.
3. **Graduated autonomy patterns** — approval gates, confidence-based routing, audit logging (~25 min)

The graduated-autonomy patterns assume familiarity with LangGraph's `StateGraph`, `TypedDict` state, and conditional edges. If you have not done M2.9 yet, see that module's super-notebook for a full walkthrough of those building blocks — Part 3 of this notebook will move briskly through them.

> **💡 Tip:** Before you change anything, copy this notebook into a folder with your name on it (e.g. `matt-grasser/`). The `TEMPLATES` folder syncs from GitHub and is read-only.

> **🔑 Before you start:** running the code below requires an API key from Anthropic or OpenAI. If you do not have one yet, see the *Getting an API Key* guide in the Module 2.10 Moodle section before going further. If you would rather not configure a paid key right now, you can still read each cell as illustration.

---

## Part 1 — Setup

### Step 1: Configure API Keys

Paste your Anthropic and/or OpenAI key in the cell below and run it. Keys stay in this kernel's memory for the session and are not persisted to disk.

In [ ]:
import os

# ✏️ Replace the placeholder values with your real API keys, then run this cell.

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-PASTE-YOUR-KEY-HERE"
os.environ["OPENAI_API_KEY"] = "sk-PASTE-YOUR-KEY-HERE"

# Verify keys are loaded (shows first/last few chars only)
for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    val = os.environ.get(key, "")
    if val and "PASTE" not in val:
        print(f"✅ {key} loaded ({val[:8]}...{val[-4:]})")
    else:
        print(f"❌ {key} not set — replace the placeholder above and re-run")


### Step 2: Verify the SDKs

In [ ]:
import importlib

packages = [
    ("anthropic", "Anthropic SDK"),
    ("openai", "OpenAI SDK"),
    ("langchain_core", "LangChain Core"),
    ("langchain_anthropic", "LangChain Anthropic"),
    ("langchain_openai", "LangChain OpenAI"),
    ("langgraph", "LangGraph"),
    ("httpx", "httpx"),
    ("dotenv", "python-dotenv"),
]

for module, name in packages:
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, "__version__", "installed")
        print(f"✅ {name}: {version}")
    except ImportError:
        print(f"❌ {name}: NOT INSTALLED")

### Step 3: Hello-world calls

If both return a sentence, your environment is ready.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(message.content[0].text)

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(response.choices[0].message.content)

---

## Part 2 — The agent loop foundation

Before you can add governance overlays — approval gates, confidence routing, audit logs — you need to understand the basic pattern they overlay. An **agent** is a language model wrapped in a small loop that lets it call **tools** (Python functions you expose with a JSON schema), see the results, and decide what to do next.

In this part you will build a basic tool-using agent with a calculator and weather lookup. **If you completed M2.9 already, you can run through this section quickly** — it is the same content as M2.9 Part 2. Either way, make sure you can trace the agent loop end-to-end before moving on to Part 3.

In [ ]:
import os
import anthropic

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

## Step 1: Define Tools

Tools are JSON schemas that tell the model what functions are available and what arguments they accept.

Let's create two simple tools: a calculator and a weather lookup.

In [ ]:
tools = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g. '(25 * 4) + 10'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Ottawa'"
                }
            },
            "required": ["city"]
        }
    }
]

## Step 2: Implement Tool Handlers

These are the actual Python functions that run when the model calls a tool.

In [ ]:
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool call to the appropriate handler."""
    if tool_name == "calculate":
        try:
            # Safety note: in production, use a proper math parser, not eval()
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"

    elif tool_name == "get_weather":
        # Simulated weather data (in a real app, call a weather API)
        fake_weather = {
            "ottawa": "☀️ 22°C, sunny",
            "toronto": "🌤️ 19°C, partly cloudy",
            "vancouver": "🌧️ 14°C, rain",
        }
        city = tool_input["city"].lower()
        return fake_weather.get(city, f"🌡️ 20°C, weather data not available for {tool_input['city']}")

    return f"Unknown tool: {tool_name}"

## Step 3: The Agent Loop

The core pattern: send a message → if the model wants to call a tool → execute it → feed the result back → repeat until the model responds with text.

This is the **agentic loop** — the model decides what to do next.

In [ ]:
def run_agent(user_message: str, max_turns: int = 5) -> str:
    """Run a simple tool-use agent loop."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # If the model just responds with text, we're done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # If the model wants to use tools, execute them
        if response.stop_reason == "tool_use":
            # Add the assistant's response (with tool_use blocks) to messages
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Calling: {block.name}({block.input})")
                    result = handle_tool_call(block.name, block.input)
                    print(f"  📎 Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Feed results back to the model
            messages.append({"role": "user", "content": tool_results})

    return "Agent reached max turns without completing."

## Step 4: Try It Out

Ask the agent questions that require tool use:

In [ ]:
# A question that requires the calculator
print("--- Calculator ---")
result = run_agent("What is 47 * 89 + 123?")
print(f"\n{result}")

In [ ]:
# A question that requires weather lookup
print("--- Weather ---")
result = run_agent("What's the weather like in Ottawa and Vancouver?")
print(f"\n{result}")

In [ ]:
# A question that requires BOTH tools
print("--- Multi-tool ---")
result = run_agent(
    "If it's 22°C in Ottawa, what is that in Fahrenheit? "
    "Also check: what's the actual weather in Toronto?"
)
print(f"\n{result}")

---

## Part 3 — Graduated autonomy and human-in-the-loop patterns

The agent loop you just built executes everything the model decides to call. In real supervisory work that is rarely acceptable — some actions carry legal weight and need a human signature before they execute. **Graduated autonomy** is the pattern: the agent's freedom to act scales with its confidence and the risk of the action. Routine work auto-executes. Ambiguous work pauses for review. High-stakes work always requires a senior signature.

In this part you will build two graduated-autonomy patterns:

- **Pattern 1 — Approval Gate.** The agent proposes a draft, a human approves (or revises) before it executes. The simplest human-in-the-loop pattern.
- **Pattern 2 — Confidence-Based Autonomy.** The agent classifies actions by risk + confidence. Routine + high-confidence auto-executes; sensitive or low-confidence escalates to a human.

Both patterns are implemented as LangGraph workflows — same `StateGraph`, `TypedDict` state, conditional edges you saw in M2.9. The new piece is the human approval node and the audit trail that records every decision.

In [ ]:
import os

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

from typing import Annotated, TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

model = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)

## Pattern 1: Approval Gate

The simplest human-in-the-loop pattern: the agent proposes an action, and a human approves or rejects it before execution.

```
[Draft] → [Human Review] → [Execute] → END
               ↓
           [Revise] → [Human Review]  (loop)
```

In [ ]:
class EmailState(TypedDict):
    """State for an email drafting workflow."""
    request: str
    messages: Annotated[list, add_messages]
    draft: str
    feedback: str
    approved: bool
    revision_count: int


def draft_email(state: EmailState) -> dict:
    """AI drafts (or revises) an email."""
    revision = state.get("revision_count", 0)

    if revision > 0 and state.get("feedback"):
        prompt = (
            f"Revise this email draft based on feedback.\n\n"
            f"Original request: {state['request']}\n\n"
            f"Current draft:\n{state['draft']}\n\n"
            f"Feedback: {state['feedback']}\n\n"
            f"Write the revised email only, no commentary."
        )
    else:
        prompt = (
            f"Draft a professional email for: {state['request']}\n\n"
            f"Write the email only, no commentary."
        )

    response = model.invoke([HumanMessage(content=prompt)])
    print(f"\n{'📝 Draft' if revision == 0 else f'✏️ Revision {revision}'}:")
    print("-" * 40)
    print(response.content)
    print("-" * 40)

    return {
        "draft": response.content,
        "messages": [response],
        "revision_count": revision + 1,
    }


def human_review(state: EmailState) -> dict:
    """Simulate human review (in production, this would pause for real input)."""
    print("\n🧑 HUMAN REVIEW")
    print("Options: [a]pprove, [r]evise with feedback, [c]ancel")

    # In a notebook, we use input() for interactive review
    choice = input("Your choice: ").strip().lower()

    if choice == "a":
        print("✅ Approved!")
        return {"approved": True, "feedback": ""}
    elif choice.startswith("r"):
        feedback = input("Feedback: ").strip()
        print(f"🔄 Sending back for revision: {feedback}")
        return {"approved": False, "feedback": feedback}
    else:
        print("❌ Cancelled")
        return {"approved": True, "draft": "[CANCELLED]", "feedback": ""}


def execute_send(state: EmailState) -> dict:
    """Execute the approved action (send email)."""
    if state.get("draft") == "[CANCELLED]":
        print("\n🚫 Email cancelled, not sent.")
    else:
        print(f"\n📤 Email sent! (after {state.get('revision_count', 0)} revision(s))")
    return state


def review_router(state: EmailState) -> str:
    """Route based on human review decision."""
    if state.get("approved", False):
        return "execute"
    return "revise"

In [ ]:
# Build the approval gate workflow
workflow = StateGraph(EmailState)

workflow.add_node("draft", draft_email)
workflow.add_node("review", human_review)
workflow.add_node("execute", execute_send)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "review")
workflow.add_conditional_edges(
    "review",
    review_router,
    {"execute": "execute", "revise": "draft"},
)
workflow.add_edge("execute", END)

email_app = workflow.compile()
print("Email workflow compiled!")

In [ ]:
# Run it! You'll be prompted to approve/revise the draft.
result = email_app.invoke({
    "request": "Invite the AI Builders Lab participants to our March 18 session. "
               "Mention it's a hands-on collaborative session using Jupyter notebooks "
               "with pre-installed AI tools. Keep it short and enthusiastic.",
    "messages": [],
    "draft": "",
    "feedback": "",
    "approved": False,
    "revision_count": 0,
})

## Pattern 2: Confidence-Based Autonomy

Instead of always asking for approval, the agent self-assesses its confidence and only asks for review when uncertain.

```
[Classify] → [High confidence] → [Auto-execute] → END
                ↓
           [Low confidence] → [Human Review] → [Execute] → END
```

In [ ]:
import json


class TaskState(TypedDict):
    """State for confidence-based routing."""
    task: str
    messages: Annotated[list, add_messages]
    classification: str
    confidence: float
    response: str
    autonomy_level: str  # "auto" or "review"


def classify_task(state: TaskState) -> dict:
    """Classify the task and assess confidence."""
    prompt = (
        f"Classify this task and rate your confidence in handling it.\n\n"
        f"Task: {state['task']}\n\n"
        f"Respond in JSON format:\n"
        f'{{"classification": "routine|complex|sensitive", '
        f'"confidence": 0.0-1.0, '
        f'"reasoning": "brief explanation"}}'
    )

    response = model.invoke([HumanMessage(content=prompt)])

    try:
        # Extract JSON from response
        text = response.content
        # Handle markdown code blocks
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        parsed = json.loads(text.strip())
    except (json.JSONDecodeError, IndexError):
        parsed = {"classification": "complex", "confidence": 0.5, "reasoning": "Could not parse"}

    classification = parsed.get("classification", "complex")
    confidence = float(parsed.get("confidence", 0.5))

    # Determine autonomy level based on confidence + classification
    if classification == "routine" and confidence >= 0.8:
        autonomy = "auto"
    elif classification == "sensitive":
        autonomy = "review"  # Always review sensitive tasks
    elif confidence >= 0.9:
        autonomy = "auto"
    else:
        autonomy = "review"

    print(f"📊 Classification: {classification} | Confidence: {confidence:.0%} | → {autonomy}")
    print(f"   Reasoning: {parsed.get('reasoning', 'N/A')}")

    return {
        "classification": classification,
        "confidence": confidence,
        "autonomy_level": autonomy,
        "messages": [response],
    }


def generate_response(state: TaskState) -> dict:
    """Generate a response to the task."""
    prompt = f"Handle this task concisely:\n\n{state['task']}"
    response = model.invoke([HumanMessage(content=prompt)])

    if state["autonomy_level"] == "auto":
        print(f"\n🤖 Auto-executing (confidence was high):")
    else:
        print(f"\n📋 Proposed response (awaiting review):")

    print(response.content[:300] + ("..." if len(response.content) > 300 else ""))

    return {"response": response.content, "messages": [response]}


def human_checkpoint(state: TaskState) -> dict:
    """Human reviews the proposed response."""
    print("\n🧑 HUMAN CHECKPOINT — This task was flagged for review.")
    choice = input("[a]pprove / [r]eject: ").strip().lower()

    if choice == "a":
        print("✅ Approved")
    else:
        print("❌ Rejected — response discarded")
        return {"response": "[REJECTED BY HUMAN]"}

    return state


def autonomy_router(state: TaskState) -> str:
    """Route based on autonomy level."""
    return state.get("autonomy_level", "review")

In [ ]:
# Build the confidence-based workflow
workflow2 = StateGraph(TaskState)

workflow2.add_node("classify", classify_task)
workflow2.add_node("generate", generate_response)
workflow2.add_node("checkpoint", human_checkpoint)

workflow2.set_entry_point("classify")
workflow2.add_edge("classify", "generate")
workflow2.add_conditional_edges(
    "generate",
    autonomy_router,
    {"auto": END, "review": "checkpoint"},
)
workflow2.add_edge("checkpoint", END)

autonomy_app = workflow2.compile()
print("Confidence-based autonomy workflow compiled!")

In [ ]:
# Test with a ROUTINE task (should auto-execute)
print("=" * 50)
print("Test 1: Routine task")
print("=" * 50)
autonomy_app.invoke({
    "task": "What is 2 + 2?",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

In [ ]:
# Test with a SENSITIVE task (should require review)
print("=" * 50)
print("Test 2: Sensitive task")
print("=" * 50)
autonomy_app.invoke({
    "task": "Draft a public statement about our company's data breach incident",
    "messages": [], "classification": "", "confidence": 0.0,
    "response": "", "autonomy_level": "",
})

---

## 💡 Key takeaways across the whole module

1. **Graduated autonomy is the pattern, not a single technique.** The agent's freedom to act scales with confidence and risk. Routine + sure → auto-execute. Sensitive or uncertain → human approval.
2. **The approval gate is just a node.** It is implemented as a LangGraph node that pauses execution and waits for a human decision. The structure is identical to any other routing decision; the difference is who answers it.
3. **Confidence-based routing is a routing function with a content check.** Read the state, decide whether the model is sure enough, branch accordingly. You write the threshold in plain Python.
4. **Audit logging is not a compliance afterthought.** Bake it into the pipeline as a first-class artifact. Every approval, rejection, and edit goes into the trail with actor + timestamp. The trail is the explanation when something needs explaining.
5. **Fail-safe by design.** Rejection at any tier stops downstream work. The default direction of failure is "do nothing." Absence of approval is never interpreted as implicit consent.

---

## You have reached the end of this super-notebook

Return to **M2.10** in the GovSpace Academy to:
- Post your Discussion Question response in the GovSpace Connect thread
- Complete the Module Quiz
- Map a multi-step workflow your team actually owns onto the three-tier pattern

This completes the agentic-AI arc in PDS. From here, the bridge into deployment work is straightforward — every supervisory pipeline you sketch from here on should pass a quick check: *where would approval gates live, what should the audit record, and which step would be the fail-safe?*